In [14]:
# Core RAG Frameworks
!pip install langchain
!pip install openai
!pip install langchain-openai
!pip install langchain-community
!pip install faiss-cpu
!pip install langchain-core
!pip install langchain-classic

In [15]:
import os
# Import the Python SDK for Colab secrets
from google.colab import userdata

# Fetch the API key from Colab secrets
openai_api_key = userdata.get('OPENAI_APIKEY')

# Set up Environmental Variables
os.environ["OPENAI_API_KEY"] = openai_api_key

In [16]:
# Import all depedencies

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.tools import create_retriever_tool
from langchain.agents import create_agent
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [17]:
# ==========================================
# Step 1: Prepare Knowledge Base (Documents)
# ==========================================
# For demonstration, creating a dummy document

sample_text = """
ACME Corp Remote Work Policy (2026):
1. Employees may work remotely up to 3 days per week with manager approval.
2. A annual stipend of $500 is provided for home office equipment setup.
3. Core working hours are 10:00 AM to 4:00 PM EST.
4. Internet reimbursement covers up to $70/month with valid receipts.
"""

with open("company_policy.txt", "w") as f:
    f.write(sample_text)

# Load and split documents into chunks
loader = TextLoader("company_policy.txt")
documents = loader.load()

In [5]:
# Load and split documents into chunks
loader = TextLoader("company_policy.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

In [6]:
# Embed and store in Vector Database (FAISS)
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [12]:
# ==========================================
# Step 2: Create the RAG Retriever Tool
# ==========================================
# This wraps the retriever into a tool that the Agent can call on demand.
retriever_tool = create_retriever_tool(
    retriever,
    name="company_policy_search",
    description="Searches and retrieves relevant policy information regarding ACME Corp."
)


In [8]:
# ==========================================
# Step 3: Initialize LLM and Prompt
# ==========================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an intelligent internal HR assistant. "
               "Use the tools provided to look up official policy facts before answering. "
               "If information is not present in the documents, state that clearly."),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [19]:
# ==========================================
# Step 4: Assemble and Run the Agent
# ==========================================
# Create Agent

agent = create_tool_calling_agent(llm, [retriever_tool], prompt)
agent_executor = AgentExecutor(agent=agent, tools=[retriever_tool], verbose=True)



In [20]:
# Example 1: Question requiring Knowledge Base lookup
response_1 = agent_executor.invoke({"input": "How much can I get reimbursed for internet?"})
print("\n--- Final Answer ---")
print(response_1["output"])

# Example 2: Question NOT requiring RAG (Agent skips tool execution)
response_2 = agent_executor.invoke({"input": "What is 15 multiplied by 12?"})
print("\n--- Final Answer ---")
print(response_2["output"])



> Entering new AgentExecutor chain...

Invoking: `company_policy_search` with `{'query': 'internet reimbursement policy'}`


ACME Corp Remote Work Policy (2026):
1. Employees may work remotely up to 3 days per week with manager approval.
2. A annual stipend of $500 is provided for home office equipment setup.
3. Core working hours are 10:00 AM to 4:00 PM EST.
4. Internet reimbursement covers up to $70/month with valid receipts.At ACME Corp, you can get reimbursed for internet expenses up to $70 per month, provided you submit valid receipts.

> Finished chain.

--- Final Answer ---
At ACME Corp, you can get reimbursed for internet expenses up to $70 per month, provided you submit valid receipts.


> Entering new AgentExecutor chain...
15 multiplied by 12 is 180.

> Finished chain.

--- Final Answer ---
15 multiplied by 12 is 180.
